## Trial code for checking trade balances

In [ ]:
#%% Checking trade balances

eu_msw["Balance"] = eu_msw["GEN"]-eu_msw["TRT"]

balance = eu_msw.groupby(by=["TIME_PERIOD"]).sum()

balance = eu_msw.loc[eu_msw["Country"]!='EU27+4']

balance = balance.groupby(by=["TIME_PERIOD","unit"])[['Balance','GEN','TRT','DSP_I_RCV_E','DSP_L_OTH','RCY']].sum()
balance = balance.reset_index()
balance = balance.loc[balance["TIME_PERIOD"]>=2001]

trade = pd.read_excel('data_downloads/Waste_shipment_data_imports_exports.xlsx', header = 8) 

trade=trade.loc[trade["General Basel Convention code -Y code"].isin(["Y18","Y46","Y47"])]
trade = trade[['Year', 'Import/export', 'Country reporting', 'Country Category',
        'Quantity in tonnes', 'To or from country', 'To or from country category',
       'Disposal and recovery code', 'General Basel Convention code -Y code']]


a = trade.groupby(by = ['Year','Import/export'])['Quantity in tonnes'].sum().reset_index()
a=pd.pivot(a,index = 'Year',columns = 'Import/export',values = 'Quantity in tonnes').reset_index()
a["Trade_Balance"]=a["Export"]-a["Import"]
a['Trade_Balance']*=1e-3
a["Export"]*=1e-3
a["Import"]*=1e-3

b = pd.merge(balance, a[["Year",'Trade_Balance']], how = 'inner', left_on = "TIME_PERIOD", right_on = "Year")

b["% covered in trade data"] = b['Trade_Balance']/b['Balance']
b['% traded wrt gen'] = b["Balance"]/b["GEN"]


trade_msw = b[["TIME_PERIOD","unit","Balance","Trade_Balance","GEN","% covered in trade data","% traded wrt gen"]]



#GEN vs TRT

test = eu_msw[['TIME_PERIOD', 'unit', 'DSP_I_RCV_E', 'DSP_L_OTH', 'GEN',
       'RCY','TRT', 'Country', 'INC%','TRT%', 'Balance']].loc[eu_msw["TIME_PERIOD"]>=2010]

test = test.loc[test["TRT%"]!=1]


test_5yrs = eu_msw[['TIME_PERIOD', 'unit', 'DSP_I_RCV_E', 'DSP_L_OTH', 'GEN',
                    'RCY','TRT', 'Country', 'INC%','TRT%', 'Balance']].loc[eu_msw["TIME_PERIOD"]>=2017]


test_2yrs = eu_msw[['TIME_PERIOD', 'unit', 'DSP_I_RCV_E', 'DSP_L_OTH', 'GEN',
                    'RCY','TRT', 'Country', 'INC%','TRT%', 'Balance']].loc[eu_msw["TIME_PERIOD"]>=2020]




#%% Analysis of balance

eu_msw["balance %"] = eu_msw['Balance']/eu_msw['GEN']
eu_msw['balance_inc%']= eu_msw['balance %']*eu_msw['INC%']
eu_msw['balance_rcy%']= eu_msw['balance %']*eu_msw['RCY%']
eu_msw['balance_dis%']= eu_msw['balance %']*eu_msw['DIS%']
eu_msw['Balance/TRT'] = eu_msw['Balance']/eu_msw['TRT']

eu_msw_5years = eu_msw.loc[eu_msw["TIME_PERIOD"]>=2017]


eu_filtered = eu_msw_5years[eu_msw_5years["Country"].isin(countries_to_group)]
eu_filtered = eu_filtered[['TIME_PERIOD', 'unit', 'DSP_I_RCV_E', 'DSP_L_OTH', 'GEN', 'RCY', 'TRT',
       'Country', 'INC%', 'TRT%', 'GEN%', 'DIS%', 'RCY%', 'Balance']]

eu_filtered['inc%*bal']=eu_filtered['INC%']*eu_filtered['Balance']
eu_filtered['rcy%*bal']=eu_filtered['RCY%']*eu_filtered['Balance']
eu_filtered['dis%*bal']=eu_filtered['DIS%']*eu_filtered['Balance']

eu_filtered = eu_filtered[['TIME_PERIOD', 'unit', 'DSP_I_RCV_E', 'DSP_L_OTH', 'GEN', 'RCY', 'TRT',
       'Country', 'inc%*bal', 'dis%*bal', 'rcy%*bal', 'Balance']]

eu_filtered = eu_filtered.groupby(["TIME_PERIOD"],as_index=False).sum()

eu_filtered['unit']='THS_T'

eu_filtered['Country']='EU27+4'

eu_filtered['inc%*bal/gen']=eu_filtered['inc%*bal']/eu_filtered['GEN'] 
eu_filtered['rcy%*bal/gen']=eu_filtered['rcy%*bal']/eu_filtered['GEN']
eu_filtered['dis%*bal/gen']=eu_filtered['dis%*bal']/eu_filtered['GEN']

#%% Example analysis for PRT, SWE

prt_data = eu_msw.loc[eu_msw['Country']=='PRT']
swe_data = eu_msw.loc[eu_msw['Country']=='SWE']

prt_data.drop(columns = ['DSP_I','RCV_E','PRP_REU','RCY_C_D','RCY_M'],inplace = True)
swe_data.drop(columns = ['DSP_I','RCV_E','PRP_REU','RCY_C_D','RCY_M'],inplace = True)


prt_trade = trade.loc[trade['Country reporting']=='Portugal']


a = prt_trade.groupby(['Year', 'Import/export', 'Country reporting', 'Country Category','General Basel Convention code -Y code'],as_index=False)['Quantity in tonnes'].sum()
a = a.loc[a['General Basel Convention code -Y code']=='Y46']


class TradeAnalysis:
    """
    self.data 
    Type: pandas DataFrame
    Contains: columns ['Year', 'Import/export', 'Country reporting', 'Country Category',
            'Quantity in tonnes', 'To or from country', 'To or from country category',
           'Disposal and recovery code', 'General Basel Convention code -Y code', 'European List of Waste code',
           'Detailed Basel Convention code or OECD decison code', 'Hazardousness',
           'Basis for classification to hazardous and non-hazardous',
           'UN hazardous code properties which render the waste hazardous',
           'Country of transit stated by a code', 'Notes']
    """
    
    def __init__(self, country_name, data):
        self.country_name = country_name
        self.data = data
        
    def total_trade_quantity(self):
        self.data
        

    def total_export_value(self):
        return sum(self.export_data.values())

    def total_import_value(self):
        return sum(self.import_data.values())

    def trade_balance(self):
        return self.total_export_value() - self.total_import_value()

    def top_export_products(self, num_products=5):
        sorted_exports = sorted(self.export_data.items(), key=lambda x: x[1], reverse=True)
        return sorted_exports[:num_products]

    def top_import_products(self, num_products=5):
        sorted_imports = sorted(self.import_data.items(), key=lambda x: x[1], reverse=True)
        return sorted_imports[:num_products]





